In [1]:
import os
#os.environ['IVOA_REGISTRY']="http://vao.stsci.edu/RegTAP/TapService.aspx"

import pyvo as vo
import warnings
# There are a number of relatively unimportant warnings that show up, so for now, suppress them:
warnings.filterwarnings("ignore", module="astropy.nddata.blocks.*")
warnings.filterwarnings("ignore", module="pyvo.utils.xml.*")
warnings.filterwarnings("ignore", module="urllib3.connectionpool.*")
import re
from astropy.table import Table
from astropy.io import fits
import astropy.coordinates as coord
coords = coord.SkyCoord.from_name("m51")

import requests
session=requests.Session()
session.verify=False

#  Set these debuglevels to 1 to see traffic details
import logging
log = logging.getLogger('urllib3')
from http.client import HTTPConnection
HTTPConnection.debuglevel = 0
from http.client import HTTPSConnection
HTTPSConnection.debuglevel = 0

import sys
sys.tracebacklimit = 0
#T Dower said:  " ForRegTAP, the preferred URL is now 
#  https://mast.stsci.edu/vo-tap/api/v0.1/registry. 
#  OAI-PMH is still on the old system."
navo_regtap = vo.dal.TAPService('https://mast.stsci.edu/vo-tap/api/v0.1/registry')
#navo_old_regtap = vo.dal.TAPService('https://vao.stsci.edu/RegTAP/TapService.aspx')
gavo_regtap = vo.dal.TAPService('https://dc.zah.uni-heidelberg.de/__system__/tap/run')
euvo_regtap = vo.dal.TAPService('https://registry.euro-vo.org/regtap/tap')
padc_regtap = vo.dal.TAPService('http://voparis-rr.obspm.fr/tap')
from pyvo import registry
from astropy.coordinates import SkyCoord
from astropy import units as u

# Registry Spring Cleaning notebook

Following up on Markus' [Confessions of a Registry Janitor](https://blog.g-vo.org/registry-a-janitor-speaks-out.html), I propose some regular checks of the metadata.  We already have checks of the validity of services, for instance, in the Operations group weather reports.  This would be compolementary.

### Check 1:  spot check numbers between different registries

What's the best way to get the current registries?  Testing one of them seems circular.  But the [RofR](https://rofr.ivoa.net) is still pointing to the old NAVO RegTAP.  OTOH, there's a bug in the new one.  

In [2]:
#result = registry.search(datamodel="regtap").to_table()
#result['ivoid','access_urls']

In [3]:
def compare( query , maxrec = 10000, count = False):
    # Currently 
    #navo_regtap = vo.dal.TAPService('https://vao.stsci.edu/RegTAP/TapService.aspx')
    #navo_new_regtap = vo.dal.TAPService('https://mast.stsci.edu/vo-tap/api/v0.1/registry')
    navo_regtap = vo.dal.TAPService('https://mast.stsci.edu/vo-tap/api/v0.1/registry')
    gavo_regtap = vo.dal.TAPService('https://dc.zah.uni-heidelberg.de/__system__/tap/run')
    euvo_regtap = vo.dal.TAPService('https://registry.euro-vo.org/regtap/tap')

    for name,regtap in [('NAVO',navo_regtap),('GAVO',gavo_regtap),('EUVO',euvo_regtap),('PADC',padc_regtap)]:
        try:
            sias = regtap.search(query, maxrec=maxrec)
            if count:
                print(f"{name} RegTAP finds {sias['cnt'][0]}")
            else:
                print(f"{name} RegTAP finds {len(sias)}")
        except Exception as e:
            print(f"{name} RegTAP gives error: {e}")       


So hardcode them for now.  Note that MAST Registry update to ADQL 2.1 reintroduced a bug with the overflow warning that should be ignored.

In [4]:
#  This syntax isn't processed correctly by NAVO's.  Use Markus' preferred.  
#compare("select count(*) as cnt from rr.capability where standard_id like '%sia%'")
compare("select * from rr.capability where standard_id like 'ivo://ivoa.net/std/sia%'")

/Users/tjaffe/space/sw/heasarc/trjaffe/pyvo/pyvo/dal/query.py:400: DALOverflowWarning: Results truncated at 493 records by service limits (you requested maxrec=10000)
  warn(f"Results truncated at {len(self.resultstable.array)} records by service limits "


NAVO RegTAP finds 493
GAVO RegTAP finds 491
EUVO RegTAP finds 492
PADC RegTAP finds 491


In [5]:
compare("select * from rr.capability where standard_id like '%hips%'")

/Users/tjaffe/space/sw/heasarc/trjaffe/pyvo/pyvo/dal/query.py:400: DALOverflowWarning: Results truncated at 533 records by service limits (you requested maxrec=10000)
  warn(f"Results truncated at {len(self.resultstable.array)} records by service limits "


NAVO RegTAP finds 533
GAVO RegTAP finds 687
EUVO RegTAP finds 687
PADC RegTAP finds 687


In [6]:
compare("select * from rr.capability where standard_id like '%cone%' and ivoid not like '%vizier%'")

/Users/tjaffe/space/sw/heasarc/trjaffe/pyvo/pyvo/dal/query.py:400: DALOverflowWarning: Results truncated at 2038 records by service limits (you requested maxrec=10000)
  warn(f"Results truncated at {len(self.resultstable.array)} records by service limits "


NAVO RegTAP finds 2038
GAVO RegTAP finds 2030
EUVO RegTAP finds 2029
PADC RegTAP finds 2030


In [7]:
compare("select * from rr.capability where standard_id like '%cone%'  and ivoid like '%vizier%'", maxrec=100000)

/Users/tjaffe/space/sw/heasarc/trjaffe/pyvo/pyvo/dal/query.py:400: DALOverflowWarning: Results truncated at 26547 records by service limits (you requested maxrec=100000)
  warn(f"Results truncated at {len(self.resultstable.array)} records by service limits "


NAVO RegTAP finds 26547
GAVO RegTAP finds 31585
EUVO RegTAP gives error: ProtocolError: ('Connection broken: IncompleteRead(14456 bytes read)', IncompleteRead(14456 bytes read))
PADC RegTAP finds 31594


## Check 1.1 look at the differences

In [8]:
query = "select * from rr.capability where standard_id like 'ivo://ivoa.net/std/sia%'"
def diffs(A, B, C, D, query):
    #  A is a tuple of ('name',service)
    resultsA = A[1].search(query).to_table()
    resultsB = B[1].search(query).to_table()
    resultsC = C[1].search(query).to_table()
    resultsD = D[1].search(query).to_table()

    idsA=[r['ivoid']+'---'+r['cap_description'] for r in resultsA ]
    idsB=[r['ivoid']+'---'+r['cap_description'] for r in resultsB ]
    idsC=[r['ivoid']+'---'+r['cap_description'] for r in resultsC ]
    idsD=[r['ivoid']+'---'+r['cap_description'] for r in resultsD ]

    diffs1 = list( set(idsA) - set(idsB) )
    print(f"Found {len(diffs1)} in {A[0]} not in {B[0]}")
    diffs2 = list( set(idsA) - set(idsC) )
    print(f"Found {len(diffs2)} in {A[0]} not in {C[0]}")
    diffs3 = list( set(idsB) - set(idsC) )
    print(f"Found {len(diffs3)} in {B[0]} not in {C[0]}")
    diffs3b = list( set(idsB) - set(idsD) )
    print(f"Found {len(diffs3)} in {B[0]} not in {D[0]}")

    diffs4 = list( set(idsB) - set(idsA) )
    print(f"Found {len(diffs4)} in {B[0]} not in {A[0]}")
    diffs5 = list( set(idsC) - set(idsA) )
    print(f"Found {len(diffs5)} in {C[0]} not in {A[0]}")
    diffs6 = list( set(idsC) - set(idsB) )
    print(f"Found {len(diffs6)} in {C[0]} not in {B[0]}")
    diffs6b = list( set(idsC) - set(idsD) )
    print(f"Found {len(diffs6b)} in {C[0]} not in {D[0]}")

    return [diffs1,diffs2,diffs3,diffs3b,diffs4,diffs5,diffs6,diffs6b]


In [9]:
difflists = diffs( ('NAVO',navo_regtap), ('GAVO',gavo_regtap), ('EUVO',euvo_regtap), ('PADC',padc_regtap), 
      "select * from rr.capability where standard_id like 'ivo://ivoa.net/std/sia%'" )

/Users/tjaffe/space/sw/heasarc/trjaffe/pyvo/pyvo/dal/query.py:404: DALOverflowWarning: Results truncated due to server limits. Consider setting a maxrec value.
  warn("Results truncated due to server limits. Consider "


Found 13 in NAVO not in GAVO
Found 12 in NAVO not in EUVO
Found 0 in GAVO not in EUVO
Found 0 in GAVO not in PADC
Found 11 in GAVO not in NAVO
Found 11 in EUVO not in NAVO
Found 1 in EUVO not in GAVO
Found 1 in EUVO not in PADC


In [10]:
print("Services in NAVO not in GAVO")
print("\n".join(difflists[0]))
print("Services in GAVO not in NAVO")
print("\n".join(difflists[3]))
print("Services in GAVO not in PADC")
print("\n".join(difflists[4]))

Services in NAVO not in GAVO
ivo://irsa.ipac/mast/scrapbook---
ivo://padc.obspm.astro/posse/q/siap---
ivo://padc.obspm.astro/srcj/q/i---
ivo://irsa.ipac/spitzer/images/level1---
ivo://vopdc.obspm/dfbs---
ivo://nasa.heasarc/skyview/planck030---
ivo://padc.obspm.astro/posse/q/cutout---
ivo://irsa.ipac/spitzer/images/level2---
ivo://padc.obspm.astro/esor/q/i---
ivo://org.gavo.dc/maidanak/res/rawframes/rawframes---
ivo://chivo/alma_fits/q/siap-alma-fits---
ivo://irsa.ipac/dss/images---
ivo://padc.obspm.astro/dfbs/q/i---
Services in GAVO not in NAVO

Services in GAVO not in PADC
ivo://org.gavo.dc/dasch/q/im---
ivo://fai.kz/schmidt_telescope_lc/q/i---
ivo://astro.upjs/upjs_img/i/i---
ivo://padc.obspm.astro-m/esor/q/i---
ivo://fai.kz/maksutov_50_telescope/q/i---
ivo://padc.obspm.astro-m/posse/q/siap---
ivo://padc.obspm.astro-m/posse/q/cutout---
ivo://iucaa/crts/siap---
ivo://padc.obspm.astro-m/dfbs/q/i---
ivo://padc.obspm.astro-m/srcj/q/i---
ivo://astro.upjs/__system__/siap2/sitewide---


The hard part is then looking at those and understanding why.  What other information would we want to look at?

### Check 2:  UCDs 

#### Check 2a:  are the UCDs valid according to astropy.io.votable.ucd.check_ucd

Start with Astropy's check_ucd():

In [11]:
from astropy.io.votable.ucd import check_ucd
query="""
  select distinct ucd, count(*) as cnt
  from rr.table_column 
  group by ucd 
  order by cnt desc
  """
result = gavo_regtap.search(query)

all_ucds = result.to_table()
culprits = []
for i,u in enumerate(all_ucds['ucd'].data):
    if not check_ucd(u):
        culprits.append((u,all_ucds['cnt'][i]))
print(f"Found {len(culprits)} invalid UCDs")
print(f"  The top 10 bad UCD values by number of instances are")
for c in culprits[0:10]: print(f"{c[0]:25}: {c[1]}")
print("")

Found 162 invalid UCDs
  The top 10 bad UCD values by number of instances are
                         : 236458
??                       : 30342
phot.flux.density;       : 672
meta.code.qual,stat.fit  : 237
vox:image_filesize       : 129
????                     : 70
image?                   : 49
phot.mag;                : 42
vox:image_mjdateobs      : 42
vox:bandpass_hilimit     : 40



Look at the ones with a semicolon ':' character:

In [12]:
colons = []
for i,u in enumerate(all_ucds['ucd'].data):
    if ":" in u:
        colons.append((u,all_ucds['cnt'][i]))
print(f"Found {len(colons)} invalid UCDs with a ':' character ")
#[ print(f"{c[0]:25}: {c[1]}") for c in colons]
for c in colons[0:10]:  print(f"{c[0]:25}: {c[1]}") 

Found 77 invalid UCDs with a ':' character 
vox:image_filesize       : 129
vox:image_mjdateobs      : 42
vox:bandpass_hilimit     : 40
vox:bandpass_id          : 40
vox:bandpass_lolimit     : 40
vox:bandpass_refvalue    : 40
vox:bandpass_unit        : 40
vox:image_naxes          : 40
vox:image_naxis          : 40
vox:image_pixflags       : 40


Now turn on the check_controlled_vocabulary flat in Astropy's check_ucd

In [13]:
culprits = []
for i,u in enumerate(all_ucds['ucd'].data):
    if not check_ucd(u,check_controlled_vocabulary=True):
        culprits.append((u,all_ucds['cnt'][i]))
print(f"Found {len(culprits)} that are not valid under UCD1+ controlled vocabulary")
print(f"  The top 10 bad UCD values by number of instances are")
for c in culprits[0:10]:  print(f"{c[0]:25}: {c[1]}") 

Found 1721 that are not valid under UCD1+ controlled vocabulary
  The top 10 bad UCD values by number of instances are
                         : 236458
??                       : 30342
error                    : 13825
code_misc                : 8538
phot_mag                 : 6291
obs.field                : 4531
fit_param                : 4505
number                   : 3090
id_number                : 2694
phot_intensity_adu       : 2512


In [14]:
bad_ucd_values = culprits

### Check 3:  authors

Have
* Last F.
* Last F., Last2 F.
* Last, F.
* F. Last, Last2. F.

At least where there are commas they are used to separate two authors, rather than "Last, F" or something.

In [15]:
names = gavo_regtap.search("select distinct role_name, count(*) as cnt from rr.res_role where base_role = 'creator' group by role_name").to_table()
names

/Users/tjaffe/space/sw/heasarc/trjaffe/pyvo/pyvo/dal/query.py:404: DALOverflowWarning: Results truncated due to server limits. Consider setting a maxrec value.
  warn("Results truncated due to server limits. Consider "


role_name,cnt
object,int32
"Guo W.-J.,Zhang Z.-X.",1
"Oh K.,Rosario D.J.",1
YuL.,1
"Santos-Sanz P.,Wilson T.G.",1
Holliman M.J.,1
"Lagrange A.-M.,Langlois M.",1
"Burstein D.,Bohlin R.C.",1
DAVILA H.,1
"Dumusque X.,Fulton B.J.",1


### Check 4:  subjects and the UAT

Note that this was wrong last year, because TJ apparently didn't know how to use the UAT correctly.

In [16]:
subjects = gavo_regtap.search("select res_subject, count(*) as cnt from rr.res_subject group by res_subject order by cnt desc").to_table()
subjects

res_subject,cnt
object,int32
visible-astronomy,8024
spectroscopy,4659
galaxies,4571
infrared-photometry,4486
photometry,4144
radial-velocity,3024
surveys,2935
redshifted,2830
variable-stars,2140


In [17]:
from pyvo.utils.vocabularies import get_vocabulary
uat = get_vocabulary("uat")["terms"]
uat_term_list = [k for k in uat.keys()]

In [18]:
culprits = []
correct = []
for i,s in enumerate(subjects['res_subject'].data):
    if s.lower() in uat_term_list:
        correct.append((s,subjects['cnt'][i]))
    else:
        culprits.append((s,subjects['cnt'][i]))
print(f"Found {len(culprits)} Registry res_subject entries that are not in the UAT and {len(correct)} that are.")
print(f"  The top 10 most frequently used bad subject values by number of occurances are\n")
for c in culprits[0:10]: print(f"{c[0]}: {c[1]}") 

Found 813 Registry res_subject entries that are not in the UAT and 425 that are.
  The top 10 most frequently used bad subject values by number of occurances are

Wide-band photometry: 1809
Survey Source: 655
Sky survey: 451
: 233
Optical astronomy: 195
Infrared astronomy: 136
extragalactic survey: 101
Radio astronomy: 94
Observational astronomy: 93
Observation: 87


### 4.1 Testing subjects at your archive

In [19]:
archive = "irsa.ipac"

subjects = gavo_regtap.search(f"""
                                select top 10 res_subject, count(*) as cnt 
                                from rr.res_subject 
                                where ivoid ilike '%{archive}%' 
                                group by res_subject 
                                order by cnt desc"""
                             ).to_table()
subjects

res_subject,cnt
object,int32
,183
extragalactic survey,101
all sky survey,58
Infrared Astronomy,41
Surveys,28
star formation,25
Milky Way disk,25
survey,24
Large Magellanic Cloud,18


In [20]:
culprits = []
correct = []
for i,s in enumerate(subjects['res_subject'].data):
    if s.lower() in uat_term_list:
        correct.append((s,subjects['cnt'][i]))
    else:
        culprits.append((s,subjects['cnt'][i]))
print(f"For {archive}:  Found {len(culprits)} Registry res_subject entries that are not in the UAT and {len(correct)} that are.")
print(f"  The top 10 bad subject values by number of instances are\n")
for c in culprits[0:10]:  print(f"{c[0]}: {c[1]}")

For irsa.ipac:  Found 9 Registry res_subject entries that are not in the UAT and 1 that are.
  The top 10 bad subject values by number of instances are

: 183
extragalactic survey: 101
all sky survey: 58
Infrared Astronomy: 41
star formation: 25
Milky Way disk: 25
survey: 24
Large Magellanic Cloud: 18
Small Magellanic Cloud: 17


### 5. Checks from Markus' article:

#### 5.1 Spatial Coverage Info

GAVO apparently puts "0/0-11" if there is no coverage information.  Hard to count how many therefore did not have any info or genuinely cover the full sky.  EuroVO apparently does not do that.  NAVO hasn't implemented stc_spatial yet.  

In [21]:
compare("select count(*) as cnt from rr.stc_spatial where cast(coverage as VARCHAR) ilike '0/0-11'",count=True)

NAVO RegTAP gives error: Exception translating ADQL: 2 unresolved identifiers: stc_spatial [l.1 c.29 - l.1 c.43], coverage [l.1 c.55 - l.1 c.63]!   - Unknown table "rr.stc_spatial" !   - Unknown column "coverage" !
GAVO RegTAP finds 1112
EUVO RegTAP finds 54
PADC RegTAP finds 1112


#### 5.2 Who needs to update the most UCDs?

Start by getting a list of publishers.

In [22]:
list_of_publishers = gavo_regtap.search("""
select distinct role_ivoid as publisher 
from rr.res_role 
where base_role = 'publisher' 
group by role_ivoid
""")
list_of_publishers.to_table()

publisher
object
ivo://3crsnapshots
ivo://ads.harvard.edu/ads
ivo://anusf.anu.au
ivo://archive.stsci.edu
ivo://archive.stsci.edu/stsci-arc
ivo://arecibo.cornell/egg
ivo://astronet.ru/organisation
ivo://au.gov.aao
ivo://badc.naoc.cn/wdc-astro


For each publisher, get the top 5 bad UCDs.  

In [23]:
for publisher in list_of_publishers['publisher'].data:
    query=f"""
      select distinct ucd, count(*) as cnt
      from rr.table_column 
      where ivoid ilike '{publisher}%'
      group by ucd 
      order by cnt desc
      """
    result = gavo_regtap.search(query)
    if len(result) > 0 and publisher != ' ':
        print(f"****    For publisher {publisher}, the invalid UCDs are:")
        for row in result.to_table():
            if not check_ucd(row['ucd']):
                print(f"    ucd {row['ucd']} appears {row['cnt']} times")
    else:
        print(f"****    For publisher {publisher}, no UCDs found.")


****    For publisher ivo://3crsnapshots, no UCDs found.
****    For publisher ivo://ads.harvard.edu/ads, no UCDs found.
****    For publisher ivo://anusf.anu.au, no UCDs found.
****    For publisher ivo://archive.stsci.edu, no UCDs found.
****    For publisher ivo://archive.stsci.edu/stsci-arc, no UCDs found.
****    For publisher ivo://arecibo.cornell/egg, no UCDs found.
****    For publisher ivo://astronet.ru/organisation, no UCDs found.
****    For publisher ivo://au.gov.aao, no UCDs found.
****    For publisher ivo://badc.naoc.cn/wdc-astro, no UCDs found.
****    For publisher ivo://cadc.nrc.ca/org, no UCDs found.
****    For publisher ivo://cds, the invalid UCDs are:
    ucd  appears 26955 times
    ucd pos.posang"unit="deg appears 1 times
****    For publisher ivo://cds.simbad, the invalid UCDs are:
    ucd  appears 131 times
    ucd pos.posang"unit="deg appears 1 times
****    For publisher ivo://cds.vizier, the invalid UCDs are:
    ucd  appears 26824 times
****    For publish

## To be expanded.  Now what to do with this?  

* Report cross-checks between registries to their admins.  
* Compile a report of issues as above and advertise at IVOA Interop's Registry (or Ops?) session.  
* Compile a report of issues found for each publisher and email them yearly to request updates.  


## Scratch 